# Homework 2


# Preparing the dataset
Use only the following columns:

'engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg'

In [7]:
COLUMNS = [
'engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg'
]

In [3]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [6]:
# EDA look into faul_effcientcy_mpg

df = pd.read_csv("../../data/car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [9]:
# Question 1

# Show missing values
df[COLUMNS].isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [10]:
# Question 2

df[COLUMNS].describe()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
count,10000.000000,9123.000000,10000.000000,10000.000000,10000.000000
mean,2268.807000,254.446016,4286.754000,1999.549500,29.997700
std,183.447355,22.844613,266.212491,14.207925,2.930245
min,1590.000000,171.000000,3320.000000,1975.000000,19.800000
25%,2150.000000,239.000000,4110.000000,1987.000000,28.000000
50%,2270.000000,254.000000,4280.000000,2000.000000,30.000000
75%,2390.000000,270.000000,4470.000000,2012.000000,31.900000
max,3110.000000,334.000000,5460.000000,2024.000000,41.200000


In [11]:
# Prepare and split the dataset
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [12]:
# Question 3

features = [col for col in COLUMNS if col != "fuel_efficiency_mpg"]
target = "fuel_efficiency_mpg"

X_train = df_train[features]
y_train = df_train[target]
X_val = df_val[features]
y_val = df_val[target]

# Option 1: Replace missing feature values with 0
model_zero = LinearRegression()
model_zero.fit(X_train.fillna(0), y_train)
pred_zero = model_zero.predict(X_val.fillna(0))
rmse_zero = round(np.sqrt(mean_squared_error(y_val, pred_zero)), 3)

# Option 2: Replace missing feature values with training-set means
train_means = X_train.mean()
model_mean = LinearRegression()
model_mean.fit(X_train.fillna(train_means), y_train)
pred_mean = model_mean.predict(X_val.fillna(train_means))
rmse_mean = round(np.sqrt(mean_squared_error(y_val, pred_mean)), 3)

print(f"RMSE with zero imputation: {rmse_zero}")
print(f"RMSE with mean imputation: {rmse_mean}")
print("Better option:", "zero imputation" if rmse_zero < rmse_mean else "mean imputation")



RMSE with zero imputation: 2.205
RMSE with mean imputation: 2.202
Better option: mean imputation


In [14]:
# Question 4 

"""Now let's train a regularized linear regression.
For this question, fill the NAs with 0.
Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
Use RMSE to evaluate the model on the validation dataset.
Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.
Which r gives the best RMSE?"""

from sklearn.linear_model import Ridge


r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
rmse_scores = {}

for r in r_values:
	model = LinearRegression() if r == 0 else Ridge(alpha=r)
	model.fit(X_train.fillna(0), y_train)
	predictions = model.predict(X_val.fillna(0))
	rmse_scores[r] = np.sqrt(mean_squared_error(y_val, predictions))
	print(f"r={r}: RMSE={rmse_scores[r]:.4f}")

best_r = min(rmse_scores, key=rmse_scores.get)
print(f"Best r: {best_r} (RMSE={rmse_scores[best_r]:.4f})")

r=0: RMSE=2.2053
r=0.01: RMSE=2.2053
r=0.1: RMSE=2.2053
r=1: RMSE=2.2053
r=5: RMSE=2.2053
r=10: RMSE=2.2053
r=100: RMSE=2.2053
Best r: 0 (RMSE=2.2053)


In [15]:
# Question 5

"""
We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
For each seed, do the train/validation/test split with 60%/20%/20% distribution.
Fill the missing values with 0 and train a model without regularization.
For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
Round the result to 3 decimal digits (round(std, 3))"""

seed_scores = []

for seed in range(10):
	rng = np.random.RandomState(seed)
	seed_idx = np.arange(len(df))
	rng.shuffle(seed_idx)

	n_train = int(len(df) * 0.6)
	n_val = int(len(df) * 0.2)

	train_df = df.iloc[seed_idx[:n_train]]
	val_df = df.iloc[seed_idx[n_train:n_train + n_val]]

	model = LinearRegression()
	model.fit(train_df[features].fillna(0), train_df[target])

	val_predictions = model.predict(val_df[features].fillna(0))
	seed_scores.append(
		np.sqrt(mean_squared_error(val_df[target], val_predictions))
	)

std_score = round(np.std(seed_scores), 3)
print("RMSE scores:", seed_scores)
print("Standard deviation:", std_score)

RMSE scores: [np.float64(2.2392041430507534), np.float64(2.2021128210837344), np.float64(2.1634197787530796), np.float64(2.204358876852931), np.float64(2.2018800943317713), np.float64(2.2457851509081417), np.float64(2.269721207951652), np.float64(2.190794599935421), np.float64(2.2166112016944184), np.float64(2.2070365928234112)]
Standard deviation: 0.029


In [16]:
# Question 6
"""Split the dataset like previously, use seed 9.
Combine train and validation datasets.
Fill the missing values with 0 and train a model with r=0.001.
What's the RMSE on the test dataset?"""

# Reuse the 60/20/20 split from seed 9
n_train = int(len(df) * 0.6)
n_val = int(len(df) * 0.2)

train_df = df.iloc[seed_idx[:n_train]]
val_df = df.iloc[seed_idx[n_train:n_train + n_val]]
test_df = df.iloc[seed_idx[n_train + n_val:]]

train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train_val = train_val_df[features].fillna(0)
y_train_val = train_val_df[target]
X_test = test_df[features].fillna(0)
y_test = test_df[target]

model = Ridge(alpha=0.001)
model.fit(X_train_val, y_train_val)

test_predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print(f"Test RMSE (seed=9, train+val combined, r=0.001): {rmse:.4f}")

Test RMSE (seed=9, train+val combined, r=0.001): 2.2358
